In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (cross_validate, StratifiedKFold,
                                     train_test_split)
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.datasets import make_classification

# ============================================================
# 1. LOAD DATA
# ============================================================
X, y = make_classification(n_samples=1000, n_features=20,
                           n_informative=10, weights=[0.9, 0.1],
                           random_state=42)
print(f"Class distribution: {np.bincount(y)}")

# ============================================================
# 2. BUILD PIPELINE (preprocessing + model together = no leakage)
# ============================================================
pipeline = Pipeline([
    ('scaler', StandardScaler()),    # Refits each CV fold
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

# ============================================================
# 3. CROSS-VALIDATION (every sample tested exactly once)
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    pipeline, X, y, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=True  # Compare train vs test for overfitting
)

print("\nCROSS-VALIDATION RESULTS")
print("=" * 50)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    train = cv_results[f'train_{metric}']
    test = cv_results[f'test_{metric}']
    gap = train.mean() - test.mean()
    print(f"{metric:12}: train={train.mean():.3f}  test={test.mean():.3f}  gap={gap:.3f}")

# ============================================================
# 4. FINAL TEST SET (touch ONCE for unbiased estimate)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\nFINAL TEST SET RESULTS")
print("=" * 50)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")